# ② 검색 — 색인 만들기 · 검색
담당: 검색 · 코드: `rag/retrieval/indexing.py`, `rag/retrieval/retrieval.py`

01 전처리의 `artifacts/chunks.jsonl` 로 Dense(KURE-v1 임베딩 → FAISS)와 BM25(Kiwi 형태소) 색인을 만들어 `artifacts/index/` 에 저장합니다.
- 임베딩은 청크를 `embedding.max_seq_length`(2,048) 토큰까지 읽습니다 — 표 청크가 잘리지 않습니다.
- BM25 는 요구사항 코드(`SFR-001`)를 한 토큰으로, 부정어 '않' · '아니' 도 남기고, 사용자 사전 · 별칭(부가세 → 부가가치세, 지체 상금 → 지체상금)으로 질문과 문서의 낱말을 맞춥니다.

검색 전에 OpenAI API로 질문을 재작성합니다(유료). 재작성한 질문으로 Dense·BM25 검색을 하고, 원본 질문에 나온 발주기관 문서 안에서만 찾습니다.

검색은 두 점수를 섞은 하이브리드입니다.
- 질문의 요구사항 코드가 그대로 든 청크를 맨 앞에 둡니다.
- 질문에 발주기관이 나오면 그 기관 문서 안에서만 찾습니다. 기관명은 발주 기관 목록 CSV(`paths.org_csv`)와 문서 표지에서 모읍니다.
  CSV(`data/data_list.csv`, UTF-8, `파일명` · `발주 기관` 열)는 담당자에게 받아 두세요 — 없으면 색인 · 검색 전에 요청하세요.
- 결과는 개수가 아니라 토큰 예산(`retrieval.token_budget`)으로 자르고, 거의 같은 청크는 하나만 남깁니다.

`.env` 에 Langfuse 키가 있으면 검색할 때마다 Langfuse Cloud 에 `retrieve-context` 로 기록됩니다 (없어도 똑같이 동작).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl, write_jsonl
from rag.retrieval.indexing import build_index
from rag.retrieval.retrieval import HybridIndex
from rag.retrieval.rewrite import rewrite_question
from rag.usage import usage_summary

cfg = load_config()

## 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
`DENSE_WEIGHT`는 0~1이며 BM25 비율은 `1 - DENSE_WEIGHT`입니다. 0은 BM25만, 1은 Dense만 사용합니다.
적용한 rewrite·검색 설정을 `artifacts/retrieval_settings.jsonl`에 저장합니다. 04는 이 파일을 읽어 같은 설정으로 검색하므로, 값을 바꾼 뒤에는 04도 다시 실행하세요.

In [ ]:
REWRITE_MODEL = cfg["rewrite"]["model"]                     # 예: "gpt-5-mini"
REWRITE_REASONING_EFFORT = cfg["rewrite"]["reasoning_effort"]  # 예: "low"
REWRITE_MAX_TOKENS = cfg["rewrite"]["max_tokens"]             # 출력·추론 토큰 한도
REWRITE_PRICES = cfg["rewrite"]["usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가
DENSE_WEIGHT = cfg["retrieval"]["dense_weight"]               # 예: 0.6 (Dense 60%, BM25 40%)

cfg["rewrite"].update(model=REWRITE_MODEL, reasoning_effort=REWRITE_REASONING_EFFORT,
                      max_tokens=REWRITE_MAX_TOKENS, usd_per_1m_tokens=REWRITE_PRICES)
cfg["retrieval"]["dense_weight"] = DENSE_WEIGHT
write_jsonl([{"rewrite": cfg["rewrite"], "retrieval": cfg["retrieval"]}], ARTIFACTS / "retrieval_settings.jsonl")
cfg["rewrite"], {"Dense": DENSE_WEIGHT, "BM25": 1 - DENSE_WEIGHT}

## 1. 색인 만들기
`임베딩에서 잘린 청크` 가 0 이면 모든 청크가 끝까지 임베딩된 것입니다.

In [ ]:
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
build_index(chunks, cfg, ARTIFACTS / "index")

## 2. 검색해 보기
위 설정 셀에서 rewrite·검색 비율을 조절한 결과를 확인하세요. 결과 길이는 `config.yaml` 의 `retrieval` (token_budget · candidate_k)이 정합니다.

In [ ]:
index = HybridIndex(ARTIFACTS, cfg)
QUESTION = "고려대학교 차세대 포털 시스템 사업의 응답 시간 요구사항은?"
rewrite = rewrite_question(QUESTION, cfg["rewrite"])
print(rewrite)
pd.DataFrame(index.search(QUESTION, query=rewrite["question"]))[["score", "title", "page", "kind", "section", "body"]]

## 3. 기관 라우팅 점검
기관명이 하나도 연결되지 않은 문서입니다. 이 문서들은 질문에 기관명이 나와도 전체 문서에서 찾습니다.

In [ ]:
unrouted = index.router.unrouted_titles()
print(f"라우팅되는 문서 {len(index.router.titles) - len(unrouted)}/{len(index.router.titles)}")
pd.DataFrame({"미라우팅 문서": unrouted})

## API 사용량
02와 04의 rewrite 호출은 모두 `artifacts/api_usage.csv`에 자동 추가됩니다. 아래 표는 누적 사용량입니다.

In [ ]:
usage_summary("rewrite")